# Fitting Conductivity with Streched Exponentials

Author: [Rylan Stutters](https://github.com/RylanDS7)

Notebook to fit the cole-cole conductivity model to stretched exponential functions under a step down response

In [1]:
import sys
import os
sys.path.append(os.path.abspath('..'))

from simpeg import *
import numpy as np
import matplotlib.pyplot as plt
from discretize import TensorMesh
from EMIP import SEInv

In [2]:
# cole cole function
def ColeColeSeigel(f, sigmaInf, eta, tau, c):
    w = 2*np.pi*f
    return sigmaInf*(1 - eta/(1 + (1j*w*tau)**c))

In [3]:
# CC SE function for step off
def ColeSEImpulsefun(eta, tau, c, time):
    return eta*c/time*((time/tau)**c)*np.exp(-(time/tau)**c)

In [4]:
# obtain predicted behavior via Hankel Transform with cos filters
# TODO Review/update
def predictSE(sigmaInf, eta, tau, c):
    from utils import DigFilter # module to be deprecated

    t = np.logspace(-6,np.log10(0.01), 41)
    wt, tbase, omega_int = DigFilter.setFrequency(t)
    f = omega_int / (2*np.pi)

    CC = ColeColeSeigel(f, sigmaInf, eta, tau, c)
    sigTCole = -DigFilter.transFiltImpulse(CC, wt, tbase, omega_int, t, tol=1e-12) # not sure why this has to be negative

    return sigTCole, t


In [ ]:
# fit SE parameters with SimPEG inversion
def fitSEparam(obsData, t):
    dtrue = obsData
    survey = SEInv.SESurvey(t)
    sim = SEInv.SEInvProblem(survey=survey)
    data_obj = data.Data(survey=survey, dobs=dtrue, standard_deviation=0)

    misfit = data_misfit.L2DataMisfit(data=data_obj, simulation=sim)
    mesh = TensorMesh([3])
    reg = regularization.Smallness(mesh)
    opt = optimization.InexactGaussNewton(maxIter=20)

    inv_prob = inverse_problem.BaseInvProblem(misfit, reg, opt)
    inv = inversion.BaseInversion(inv_prob)

    m0 = np.array([1.0, 1.0, 1.0])  # initial guess

    mrec = inv.run(m0)

    print("Recovered model:", mrec)

In [6]:
eta1, tau1, c1 = 0.1, 1e-3, 0.7

obsSigma, t = predictSE(1, eta1, tau1, c1)

fitSEparam(obsSigma, t)

plt.loglog(t*1e3, obsSigma, 'k', lw=1)

c:\Users\ryds1\OneDrive\Desktop\GIF\2026-Stutters-EMIP\utils\DigFilter.py:5: FutureWarning: Importing `SimPEG` is deprecated. please import from `simpeg`.
  from SimPEG import utils
INFO: simpeg.InvProblem will set Regularization.reference_model to m0.
INFO: 
simpeg.InvProblem is setting bfgsH0 to the inverse of the reg.deriv2.
using the default solver Pardiso with the 'is_symmetric=True` option set.




Running inversion with SimPEG v0.25.0
============================ Inexact Gauss Newton ============================
  #     beta     phi_d     phi_m       f      |proj(x-g)-x|  LS    Comment   
-----------------------------------------------------------------------------
   0  1.00e+00  2.56e+08  0.00e+00  2.56e+08                                 


c:\Users\ryds1\OneDrive\Desktop\GIF\2026-Stutters-EMIP\EMIP\SEInv.py:26: RuntimeWarning: invalid value encountered in power
  return eta * c / t * ((t / tau)**c) * np.exp(-(t / tau)**c)


ValueError: The `SEInvProblem.dpred()` method returned an array that contains `nan`s and/or `inf`s.